# Local RAG Pipeline with HuggingFace Embeddings (Python 3.12)
This notebook demonstrates how to:
- Read local `.txt` and `.pdf` files
    - This is good enough for 'playing' but production would need  many more examples of files AND longer files.
    - A single service would not be enough
- Chunk text for embedding

In [1]:
%pip install --upgrade pip

%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from config import settings

In [3]:
docs_path: Path = settings.DATA_DIR
processed_path: Path = settings.PROCESSED_FILE

print(settings)

Settings(
  BASE_PATH=.
  DATA_DIR=data_sources
  API_KEY=***
  PROCESSED_FILE=processed_data.txt
  DEBUG=False
  SENTENCE_TRANSFORMER_MODEL=all-MiniLM-L6-v2
  SENTENCE_TRANSFORMER_CHUNK_SIZE=500
  SENTENCE_TRANSFORMER_OVERLAP_SIZE=50
)


## Base Strategy Interface

In [4]:
from pathlib import Path
from abc import ABC, abstractmethod


class FileHandler(ABC):
    @abstractmethod
    def can_handle(self, file_path: Path) -> bool: ## How robust is this?
        pass

    @abstractmethod
    def extract_text(self, file_path: Path) -> str: ## How robust is this? How do we know the text is extracted correctly?
        pass

## The DocumentLoader

In [5]:
class DocumentLoader:
    def __init__(self):
        self.handlers: list[FileHandler] = []

    def register_handler(self, handler: FileHandler):
        self.handlers.append(handler)

    def extract_text(self, file_path: Path) -> str:
        for handler in self.handlers:
            if handler.can_handle(file_path): ## check if the handler can handle the file type
                ## if yes, call the extract_text method of the handler
                return handler.extract_text(file_path)
        raise ValueError(f"No handler for file type: {file_path.suffix}")

## Text Handler

In [6]:
class TxtHandler(FileHandler):
    def can_handle(self, file_path: Path) -> bool:
        return file_path.suffix.lower() == ".txt"

    def extract_text(self, file_path: Path) -> str:
        return file_path.read_text(encoding="utf-8")

## PDF Handler
Note that PDFs can be encoded with text or as 'images' of text. This class checks and handles both cases

In [7]:
import PyPDF2
from pdf2image import convert_from_path
import pytesseract


class PdfHandler(FileHandler):
    def can_handle(self, file_path: Path) -> bool:
        return file_path.suffix.lower() == ".pdf"

    def is_text_based(self, file_path: Path) -> bool:
        try:
            with open(file_path, "rb") as f:
                reader = PyPDF2.PdfReader(f)
                return any(page.extract_text() for page in reader.pages)
        except:
            return False

    def extract_text(self, file_path: Path) -> str:
        if self.is_text_based(file_path):
            with open(file_path, "rb") as f:
                reader = PyPDF2.PdfReader(f)
                return "\n".join(page.extract_text() or "" for page in reader.pages)
        else:
            pages = convert_from_path(file_path)
            return "\n".join(pytesseract.image_to_string(p) for p in pages)

In [8]:
import re

def clean_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'Page \d+', '', text, flags=re.IGNORECASE)
    text = ''.join(c for c in text if c.isprintable())
    return text.strip()

In [9]:
loader = DocumentLoader()
loader.register_handler(TxtHandler())
loader.register_handler(PdfHandler())

In [10]:
from dataclasses import dataclass
from typing import List, Dict

@dataclass
class TextFile:
    filename: str
    content: str


all_texts: List[TextFile] = []

In [11]:
for file in docs_path.rglob("*"): ## Is this recursive? Yes, rglob is recursive
    try:
        content = loader.extract_text(file)
        cleaned = clean_text(content)
        all_texts.append(TextFile(filename=file.name, content=cleaned))
    except Exception as e:
        print(f"Error with {file.name}: {e}")

## Check the documents have been processed

In [12]:
print(f"Extracted {len(all_texts)} documents.")

Extracted 5 documents.


In [13]:
print(all_texts[1])

TextFile(filename='LHH Values Audit Exercise.pdf', content='Values Audit Values are powerful principles or qualifications that underpin our actions and our opinions about events and people. Sometimes they are called ‘critical needs’ . Values change significantly over the years and it’s important to be clear about them and to better assess how they can be met. Decisions we make often reflect what’s important in our lives. If there’s a conflict between a decision and your own values, very often this conflict can contribute to both personal and career dissatisfaction and unhappiness. Sources of strong dissatisfaction often occur when we have cherished values that are not currently being acted on or when we have two strong values that are in conflict, such as a desire for achievement and leisure. Often we’re not fully aware of values that drive us, especially if they have always been satisfied. 2 © LHH. All rights reserved Values Audit Exercise Directions 1. Review each of the values liste

## Write to file(s)
### Write to .txt file for validation

In [ ]:
from datetime import date

with open(processed_path, "w", encoding="utf-8") as f:  ## Open the file in write mode
    for doc in all_texts: ## Loop through the documents
        ## Maybe add the meatadata as JSON?
        f.write(f"__META__FILE__NAME: {doc.filename}\n")
        f.write(f"__CONTENT__: {doc.content}\n")
        f.write(f"__META__DATE: {date.today().isoformat()}\n")
        f.write(f"__META__SOURCE: internal_docs")

        if settings.DEBUG:
            f.write("\n" + "="*80 + "\n\n") ## Document separator
        else:
            f.write("\n")

### Chunk and add to .txt ready for next step

In [ ]:
from datetime import datetime
import uuid

chunk_id = uuid.uuid4()

chunked_file_path = settings.BASE_PATH / "chunked_data.txt"


def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks


with open(chunked_file_path, "w", encoding="utf-8") as cf:
    for doc in all_texts:
        chunks = chunk_text(
            doc.content,
            settings.SENTENCE_TRANSFORMER_CHUNK_SIZE,
            settings.SENTENCE_TRANSFORMER_OVERLAP_SIZE,
        )
        for i, chunk in enumerate(chunks): ## Metadata maybe better as JSON?
            cf.write(f"__META__FILE__NAME: {doc.filename}\n")
            f.write(f"__ID__: {chunk_id}\n")
            cf.write(f"__META__CHUNK: {i + 1}\n")
            cf.write(f"__META__DATE: {datetime.today().isoformat()}\n")
            cf.write(f"__CONTENT__: {chunk.strip()}\n")
            if settings.DEBUG:
                cf.write("\n" + "=" * 80 + "\n\n")
            else:
                cf.write("\n")